In [1]:
%reload_ext autoreload
%autoreload 2

import sys

sys.path.append("..")

In [2]:
import ast
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm import tqdm

from ltr_utility.clustering import QueryPartition, AggrMode
from ltr_utility.dataset.load_dataset import load_by_query_dataset, DatasetName
from ltr_utility.model_selection import save_dict_to_json

base_path = Path("../datasets")

# Every retrain notebook reads <dataset>/results/query_similarity.json, so the
# partitions must be written into that exact directory.
results_path = Path("../experiments/query_based")

Q_PER_MODEL = [1, 2, 4, 6, 8, 10, 50, 100, 250, 500]
# MQ2008 and FINDHR retain fewer queries after filtering, so their sweep stops earlier.
Q_PER_MODEL_SMALL = [1, 2, 4, 6, 8, 10, 50, 100]

LETOR_FEATURES = [20, 21, 22, 23, 24]
WEB10K_FEATURES = [105, 106, 107, 108, 109]


def sweep(dt, q_per_model, out_dir, features=None, aggregated=False):
    """Group similar queries for each batch size and save the lookup as JSON."""
    sim_queries = {
        qxm: QueryPartition(n_elements=qxm, agg_mode=AggrMode.FEATURE,
                            features=features).fit_cosine(dt, aggregated=aggregated)
        for qxm in tqdm(q_per_model)
    }
    save_dict_to_json(sim_queries, results_path / out_dir / "results/query_similarity.json")

## Notebook purpose

This notebook precomputes query-similarity partitions for the query-based experiments. For each dataset, it sweeps `q_per_model`, groups similar queries with `QueryPartition.fit_cosine`, and saves a JSON mapping that downstream model-selection scripts can reuse.

`AggrMode.FEATURE` represents each query through feature-distribution statistics before cosine grouping. When a `features` list is provided, only that dataset-specific descriptor slice is used to build the query embeddings.


# MQ 2007

The MQ2007 block builds partitions for several target batch sizes. `qxm` is the maximum number of similar queries assigned to one group; smaller values create more specialized query batches, while larger values merge more queries together.


In [3]:
sweep(load_by_query_dataset(base_path, DatasetName.MQ),
      Q_PER_MODEL, "MQ2007", features=LETOR_FEATURES)

Loading train dataset from cache.
Train data loaded from cache.
Loading valid dataset from cache.
Valid data loaded from cache.
Loading test dataset from cache.
Test data loaded from cache.
---- MQ2007 2007 loaded ----
Filtered out 1 queries with fewer than 10 documents.
---- discard_minority_groups 10 queries ----
---- Get first 500 queries ----
---- max_item 400 -(determistic!) ----


100%|██████████| 10/10 [00:00<00:00, 33.52it/s]


# MQ 2007 LIST

This section repeats the same similarity workflow for the listwise MQ2007 variant. The cell below loads the listwise dataset, computes query partitions, and stores them under the experiment results directory.


In [4]:
sweep(load_by_query_dataset(base_path, DatasetName.MQ2007LIST),
      Q_PER_MODEL, "MQ2007LIST", features=LETOR_FEATURES)

Loading train dataset from source file.
Train data loaded and saved to cache.
Loading valid dataset from source file.
Valid data loaded and saved to cache.
Loading test dataset from source file.
Test data loaded and saved to cache.
---- MQ2007 2007 LIST loaded ----
Filtered out 0 queries with fewer than 10 documents.
---- discard_minority_groups 10 queries ----
---- Get first 500 queries ----
---- max_item 400 -(determistic!) ----
---- Update the listwise ranking target  ----


100%|██████████| 10/10 [00:00<00:00, 19.91it/s]


# MQ 2008

MQ2008 uses the same LETOR feature slice as MQ2007 (`20`-`24`) for query-level similarity. The sweep stops at `100` because this dataset has fewer retained queries after filtering.


In [5]:
sweep(load_by_query_dataset(base_path, DatasetName.MQ2008),
      Q_PER_MODEL_SMALL, "MQ2008", features=LETOR_FEATURES)

Loading train dataset from cache.
Train data loaded from cache.
Loading valid dataset from cache.
Valid data loaded from cache.
Loading test dataset from cache.
Test data loaded from cache.
---- MQ2007 2008 loaded ----
Filtered out 403 queries with fewer than 10 documents.
---- discard_minority_groups 10 queries ----
---- Get first 381 queries ----
---- max_item 400 -(determistic!) ----


100%|██████████| 8/8 [00:00<00:00, 45.65it/s]


# MQ 2008 LIST

The listwise MQ2008 block mirrors the MQ2007 listwise flow: load the transformed labels, aggregate each query over the selected descriptor features, group queries by cosine similarity, and save one JSON file for all tested group sizes.


In [6]:
sweep(load_by_query_dataset(base_path, DatasetName.MQ2008LIST),
      Q_PER_MODEL, "MQ2008LIST", features=LETOR_FEATURES)

Loading train dataset from source file.
Train data loaded and saved to cache.
Loading valid dataset from source file.
Valid data loaded and saved to cache.
Loading test dataset from source file.
Test data loaded and saved to cache.
---- MQ2007 2008 LIST loaded ----
Filtered out 0 queries with fewer than 10 documents.
---- discard_minority_groups 10 queries ----
---- Get first 500 queries ----
---- max_item 400 -(determistic!) ----
---- Update the listwise ranking target  ----


100%|██████████| 10/10 [00:00<00:00, 20.27it/s]


# WEB10K

WEB10K has a different feature schema, so the query descriptor slice changes to `105`-`109`. The output file is the query-partition lookup used by WEB10K query-based experiments.


In [7]:
sweep(load_by_query_dataset(base_path, DatasetName.WEB10),
      Q_PER_MODEL, "WEB", features=WEB10K_FEATURES)

Loading train dataset from source file.
Train data loaded and saved to cache.
Loading valid dataset from source file.
Valid data loaded and saved to cache.
Loading test dataset from source file.
Test data loaded and saved to cache.
---- WEB10K loaded ----
Filtered out 187 queries with fewer than 10 documents.
---- discard_minority_groups 10 queries ----
---- Get first 500 queries ----
---- max_item 400 -(determistic!) ----


100%|██████████| 10/10 [00:00<00:00, 29.16it/s]


# Yahoo

Yahoo does not pass an explicit feature slice, so `QueryPartition` aggregates all available features for each query before computing cosine-based groups.


In [8]:
# No feature slice: QueryPartition aggregates all available features.
sweep(load_by_query_dataset(base_path, DatasetName.YAHOO), Q_PER_MODEL, "YAHOO")

Loading train dataset from source file.
Train data loaded and saved to cache.
Loading valid dataset from source file.
Valid data loaded and saved to cache.
Loading test dataset from source file.
Test data loaded and saved to cache.
---- YAHOO loaded ----
Filtered out 358 queries with fewer than 10 documents.
---- discard_minority_groups 10 queries ----
---- Get first 500 queries ----
---- max_item 400 -(determistic!) ----


100%|██████████| 10/10 [00:02<00:00,  4.52it/s]


# FINDHR

FindHR starts from job metadata rather than an `LtrDataset`. The code converts required skills and language skills into a binary job-feature matrix, then calls `fit_cosine(..., aggregated=True)` because each row is already an aggregated query/job representation.


In [9]:
jobs = pd.read_csv(base_path / "FindHR/20250530_JDS_pre.csv",
                   usecols=["id_j", "skills_req_j", "lang_skills_j"],
                   converters={"id_j": int,
                               "skills_req_j": ast.literal_eval,
                               "lang_skills_j": ast.literal_eval})

# One binary column per distinct skill / language, jobs as rows.
job_features = pd.concat(
    [pd.DataFrame(MultiLabelBinarizer().fit_transform(jobs[col]), index=jobs["id_j"])
     for col in ("skills_req_j", "lang_skills_j")],
    axis=1,
)

# aggregated=True: each row is already one query (job), not a document set.
sweep(job_features.values, Q_PER_MODEL_SMALL, "FINDHR", aggregated=True)

100%|██████████| 8/8 [00:00<00:00, 2251.52it/s]
